In [ ]:
"""
so what we wnat to do here 
is actually construct a means for us to use our previously created models 
and use them to try and construct a "Pneumonia likely " / triage score. 
how is this going to do this?
grab our various best models (there are a couple of considerations I need to make). 
mostly, I need to grab our RAW CNN (will need to retrain him), our IMAGENET-112 finetuned, and our CHEXPERT finetuned models
then for our MLP, we can try a couple of different combinations for the NLP
I also want to create a version of each for abnormal labels
which means we need to create 2 df's -- one for pneumonia and one for abnormal. 
here's the thing -- I don't think I have the labelled data necessary to be able to replicate the CHexpert paper
its a matter of data labelling -- I don't have clear labels. Heck, even just indentifying that a paitent has pneumonia from our labelled data
is not a sure thing. 
But if we have 2 results
Pneumonia and Abnormal
and we can compare them
we can see which one showcase what I want to better. 
in reality, I am really interested in both, so I will probably do both. 
The triage score then becomes "how likely are they to be sick given all of our notes". 
if I was better at identifying WHAT was wrong, I could give them an additional ranking as well. 
however, I don't have access to that sort of information... yet. Might be worth looking into in the future. 
There's gotta be a way we can try to account for that in our stuff, right?
so lets go about rewriting everything to do waht I want them to do and go from there. 
"""

In [6]:
import torch
from NoteBooks.fineTUneCNNTime import * # I don't think that quite works. 
from NoteBooks.MLP_pneumonia import * # need the class form here as well. 
import pandas as pd

ModuleNotFoundError: No module named 'NoteBooks'

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
projections_path = "../IU-xray/indiana_projections.csv"
reports_path = "../IU-xray/indiana_reports.csv"

In [5]:
# Model selection parameters. 
MLP_type = "Standard"
CNN_type = "ImageNetFineTuned"

In [ ]:
# Load the selected models. 
print("Loading the models ... ")

model_mlp = MLP(input_nodes) # figure out the imports for this in a second
model_mlp = model_mlp.to(device)
if MLP_type == "Standard":
    checkpoint = torch.load(f"../Models/BestMLP's/Pneumonia/Best_MLP_Pneumonia.pth")
    model_mlp.load_state_dict(checkpoint['model_state_dict'])
else:
    print("this is not supported yet")

if CNN_type == "ImageNetFineTuned":
    model_CNN = FinetunedDenseNet()
    model_CNN = model.to(device)
    checkpoint = torch.load(f"../Models/BestCNN's/Pneumonia/BestDenseNetPneumonia/Best_densenet_pneumonia.pth")
    model_CNN.load_state_dict(checkpoint["model_state_dict"])

else:
    print("this is not supported yet")

In [ ]:
# this is the DF thing that want, kind of. 
import re

# Shared negation checker
def simple_negation_check(text, target_terms):
    """Return True if any target term is present and not negated."""
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    
    if not any(term in text_lower for term in target_terms):
        return False
    
    neg_patterns = [
        r'\bno\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bwithout\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bfree\s+of\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnegative\s+for\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
        r'\bnot\s+(?:\w+\s+){0,5}(?:' + '|'.join(target_terms) + r')\b',
    ]
    for pat in neg_patterns:
        if re.search(pat, text_lower):
            return False
    return True

PNEUMONIA_TERMS = ['pneumonia', 'consolidation', 'infiltrate', 'airspace disease', 'opacity']

def build_master_df(projections_path, reports_path, mode='pneumonia'):
    """
    Build master dataframe for CNN training.
    
    Args:
        projections_path: path to projections CSV
        reports_path: path to reports CSV
        mode: 'pneumonia' for pneumonia surrogate label
              'abnormal' for abnormal vs normal label
    
    Returns:
        DataFrame with columns ['filename', 'label']
    """
    print("This is the projections path ", projections_path)
    proj = pd.read_csv(projections_path)
    rep = pd.read_csv(reports_path)
    
    proj = proj[proj["projection"] == "Frontal"]
    
    if mode == 'pneumonia':
        # Create pneumonia label using free-text negation check
        rep["label"] = rep.apply(
            lambda row: 1 if simple_negation_check(
                f"{row.get('findings','')} {row.get('impression','')}",
                PNEUMONIA_TERMS
            ) else 0,
            axis=1
        )
        df = proj.merge(rep[['uid', 'label']], on='uid', how='inner')
        print(f"Pneumonia mode: {df['label'].sum()} positives / {len(df)} total ({df['label'].mean():.1%})")
        
    elif mode == 'abnormal':
        # Abnormal = anything not explicitly "normal"
        rep["label"] = rep["Problems"].apply(
            lambda x: 0 if str(x).lower().strip() == 'normal' else 1
        )
        df = proj.merge(rep[['uid', 'label']], on='uid', how='inner')
        print(f"Abnormal mode: {df['label'].sum()} positives / {len(df)} total ({df['label'].mean():.1%})")
        
    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'pneumonia' or 'abnormal'.")
    
    return df(['uid', 'filename', 'findings', 'impression', 'indication', 'Problems', 'label'])


# Backward compatibility aliases
def build_master_df_for_pneumonia(projections_path, reports_path):
    """Legacy wrapper for pneumonia mode."""
    return build_master_df(projections_path, reports_path, mode='pneumonia')

def build_master_df_for_abnormal(projections_path, reports_path):
    """Legacy wrapper for abnormal mode."""
    return build_master_df(projections_path, reports_path, mode='abnormal')

In [ ]:
# now I need to create a validation loader. 
# how is this gonna work???
# well. 
# we need to create a new dataloader based on if its pneumonia or abnormal. 
df = build_master_df(projections_path, reports_path, mode="pneumonia")